# Airflow Workflow Design
**Modules 9-10 · Advanced** · part 5 of 8 of *Building Automated Data Pipelines with Spark, dbt and Airflow* (UK · USA · Mauritius job-market aligned)

Each notebook is **self-contained**: run top to bottom. Needs Python 3.9+ and Java 8/11/17 for PySpark (not needed for 01, 05, 08).

**Series:**
- `01_DFD_Data_Flow_Diagrams.ipynb`: Modules 1-2 · Beginner
- `02_Modular_Pipeline.ipynb`: Modules 3-4 · Beginner to Intermediate
- `03_Connectors_and_Integration.ipynb`: Modules 5-6 · Intermediate
- `04_SCD2_Spark_and_dbt.ipynb`: Modules 7-8 · Intermediate to Advanced
- `05_Airflow_Workflow_Design.ipynb`: Modules 9-10 · Advanced
- `06_Capstone_End_to_End.ipynb`: Module 11 · Advanced
- `07_Advanced_Performance_Testing_CICD.ipynb`: Beyond the course · Advanced
- `08_Career_UK_USA_Mauritius.ipynb`: Job-market alignment · All levels

## Setup

In [ ]:
import os, sys, json, time, shutil, logging, sqlite3, ast
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
BASE = Path("pipeline_lab"); BASE.mkdir(exist_ok=True)

# PART 5: Workflow design and production (Modules 9–10) · Advanced

**Workflow design principles**
- **Idempotent**: re-running a task for the same date gives the same result (no duplicates)
- **Atomic**: write to temp, then swap
- **Retries with backoff**, **timeouts**, **SLAs**, **alerts on failure**
- **Dependencies explicit** (DAG), **small tasks**, **no heavy logic in the DAG file**
- **Backfill-friendly**: use the logical date (`{{ ds }}`), not `now()`
- **`max_active_runs=1`** when tasks share state (e.g. SCD2 dim)

Below is a production-style Airflow 2.x DAG: ingest in parallel → Spark SCD2 → dbt build → data-quality gate → notify.

In [ ]:
(BASE / "dags").mkdir(exist_ok=True)
DAG_FILE = BASE / "dags/customer_360_pipeline.py"
DAG_FILE.write_text('''
from __future__ import annotations
import logging
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.empty import EmptyOperator
from airflow.operators.python import PythonOperator
from airflow.utils.task_group import TaskGroup
from airflow.utils.trigger_rule import TriggerRule

log = logging.getLogger(__name__)
PROJECT = "/opt/airflow/pipeline_lab"          # mounted path in your Airflow image

def notify_failure(context):
    ti = context["task_instance"]
    # Replace with Slack / MS Teams / PagerDuty / email callback
    log.error("ALERT dag=%s task=%s run=%s", ti.dag_id, ti.task_id, context["run_id"])

def quality_gate(**context):
    # Read row counts from the metrics file written by the Spark job and fail loudly if bad
    log.info("running DQ gate for %s", context["ds"])

default_args = {
    "owner": "data-engineering",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,
    "execution_timeout": timedelta(hours=1),
    "on_failure_callback": notify_failure,
}

with DAG(
    dag_id="customer_360_pipeline",
    description="Spark ingest + SCD2, dbt models/tests, DQ gate",
    start_date=datetime(2026, 1, 1),
    schedule="0 2 * * *",
    catchup=False,
    max_active_runs=1,
    default_args=default_args,
    tags=["spark", "dbt", "customer360"],
) as dag:

    start = EmptyOperator(task_id="start")

    with TaskGroup("ingest") as ingest:
        for src in ["crm_customers", "orders", "web_events"]:
            BashOperator(
                task_id=f"ingest_{src}",
                bash_command=f"spark-submit {PROJECT}/jobs/ingest.py --source {src} --run-date {{{{ ds }}}}",
                sla=timedelta(minutes=30),
            )

    scd2 = BashOperator(
        task_id="spark_scd2_dim_customer",
        bash_command=f"spark-submit {PROJECT}/jobs/scd2_customer.py --run-date {{{{ ds }}}}",
    )

    dbt_build = BashOperator(
        task_id="dbt_build",
        bash_command=f"cd {PROJECT}/dbt_project && dbt build --profiles-dir . --vars \\'{{run_date: {{{{ ds }}}}}}\\'",
    )

    dq = PythonOperator(task_id="quality_gate", python_callable=quality_gate)

    done = EmptyOperator(task_id="done", trigger_rule=TriggerRule.ALL_SUCCESS)

    start >> ingest >> scd2 >> dbt_build >> dq >> done
''')

tree = ast.parse(DAG_FILE.read_text())           # syntax validation without needing Airflow installed
print("DAG file is valid Python. Operators used:",
      sorted({n.func.id for n in ast.walk(tree) if isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and n.func.id.endswith("Operator")}))

**Run Airflow locally (optional)**
```bash
# Docker (recommended)
curl -LfO https://airflow.apache.org/docs/apache-airflow/stable/docker-compose.yaml
mkdir -p dags logs plugins && echo "AIRFLOW_UID=$(id -u)" > .env
# copy pipeline_lab/dags/customer_360_pipeline.py into ./dags
docker compose up airflow-init && docker compose up -d      # UI at http://localhost:8080
airflow dags test customer_360_pipeline 2026-09-02          # run one date end-to-end
```
**Production-readiness checklist (memorise for interviews)**
- Secrets in a backend (AWS Secrets Manager, Azure Key Vault, Airflow Connections), never in code
- Use `KubernetesPodOperator`/Databricks/EMR/Dataproc operators for heavy Spark instead of running on the scheduler
- Sensors/Datasets for event-driven triggers; pools to limit concurrency
- DAG-level tests (`DagBag` import test, no cycles) in CI
- Alerting + runbooks; dashboards for duration, failures, freshness